# Chat Templates

## Introduction

A **chat template** turns a list of chat messages into the single string a language model actually reads. Models don't see "messages". They see one long sequence of tokens. The template sets how that sequence is built: where each message starts and ends, which role (system, user, assistant, tool) each message belongs to, and how the model is told it's its turn to reply.

Each instruct model is fine-tuned on one specific format. If you give it the same format at inference, it acts like an assistant. If you give it a different format, or none, the output gets worse: rambling, ignored instructions, or the model writing the user's lines too.

### Base vs. Instruct Models

- **Base model** (e.g. `SmolLM2-135M`): trained only to predict the next token of raw text. It continues documents and doesn't hold a conversation.
- **Instruct model** (e.g. `SmolLM2-135M-Instruct`): fine-tuned on conversations in one fixed template, so it follows instructions, takes turns, and can use tools.

### What a Template Looks Like

Take this conversation:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
]
```

**ChatML** (SmolLM2, Qwen 2) renders it as:

```text
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
```

The last line, `<|im_start|>assistant`, is the **generation prompt**. It tells the model to write the assistant's reply next. Other model families use other markers: Llama 3 uses `<|start_header_id|>…<|end_header_id|>` and `<|eot_id|>`, and Mistral uses `[INST] … [/INST]`.

### Applying a Template in Code

You shouldn't write templates by hand. The model's tokenizer stores its own template (a Jinja file), and `apply_chat_template` renders it:

```python
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

prompt = tok.apply_chat_template(
    messages,
    tokenize=False,             # return a string instead of token ids
    add_generation_prompt=True, # add the assistant cue (inference)
)
print(prompt)
```

- Use `add_generation_prompt=True` at **inference**, when you want the model to answer.
- Use `add_generation_prompt=False` for **training data**, where the assistant reply is already in the messages.
- Run `print(tok.chat_template)` to see the raw template for any model.

### Why It Matters

- **Correctness:** the model only behaves as intended in the format it was trained on.
- **Fine-tuning:** training data has to be formatted with the same template you'll use at inference.
- **Advanced features:** tool calling (`tools=[...]`), multimodal inputs, and multi-turn history are all encoded through the template.

---

## Best Practices

### General Guidelines

When working with chat templates, follow these key practices:

1. **Consistent Formatting:** Always use the same template format throughout your application.
2. **Clear Role Definition:** Clearly specify roles (`system`, `user`, `assistant`, `tool`) for each message.
3. **Context Management:** Be mindful of token limits when maintaining conversation history.
4. **Error Handling:** Include proper error handling for tool calls and multimodal inputs.
5. **Validation:** Validate message structure before sending it to the model.

### Common Pitfalls to Avoid

- Mixing different template formats in the same application
- Exceeding token limits with long conversation histories
- Not properly escaping special characters in messages
- Forgetting to validate input message structure
- Ignoring model-specific template requirements

In [12]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceTB/smoltalk", "everyday-conversations")

In [13]:
def convert_to_chatml(example):
    return {
        "messages": [
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

In [14]:
dataset['train'][0]

{'full_topic': 'Travel/Vacation destinations/Beach resorts',
 'messages': [{'content': 'Hi there', 'role': 'user'},
  {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
  {'content': "I'm looking for a beach resort for my next vacation. Can you recommend some popular ones?",
   'role': 'user'},
  {'content': "Some popular beach resorts include Maui in Hawaii, the Maldives, and the Bahamas. They're known for their beautiful beaches and crystal-clear waters.",
   'role': 'assistant'},
  {'content': 'That sounds great. Are there any resorts in the Caribbean that are good for families?',
   'role': 'user'},
  {'content': 'Yes, the Turks and Caicos Islands and Barbados are excellent choices for family-friendly resorts in the Caribbean. They offer a range of activities and amenities suitable for all ages.',
   'role': 'assistant'},
  {'content': "Okay, I'll look into those. Thanks for the recommendations!",
   'role': 'user'},
  {'content': "You're welcome. I hope you find

In [15]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

In [16]:
def apply_template(example):
    text = tok.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        'text': text
    }

In [17]:
dataset = dataset.map(apply_template, remove_columns=dataset['train'].column_names)

In [18]:
def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=512)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

Map: 100%|██████████| 119/119 [00:00<00:00, 2299.05 examples/s]


In [19]:
from transformers import AutoModelForCausalLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

args = TrainingArguments(
    output_dir="results/smollm2-chat",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    learning_rate=5e-5,
    logging_steps=50,
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=collator,
)

trainer.train()

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1252.98it/s]


Epoch,Training Loss,Validation Loss
1,0.884737,0.897307


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.98it/s]


TrainOutput(global_step=565, training_loss=0.9064944782088288, metrics={'train_runtime': 119.9797, 'train_samples_per_second': 18.837, 'train_steps_per_second': 4.709, 'total_flos': 361543752308736.0, 'train_loss': 0.9064944782088288, 'epoch': 1.0})

In [20]:
chat = [{"role": "user", "content": "Can you recommend a beach for families?"}]
inputs = tok.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt", return_dict=True)
out = model.generate(**inputs, max_new_tokens=100)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\transformers\generation\utils.py:2950: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Absolutely, I'd be happy to help you with that. Here are some beach destinations that are perfect for families:

1. **Coral Gables, New York**: This charming coastal town is known for its beautiful beaches, historic landmarks, and a lively atmosphere. It's a great place to relax and enjoy the sun.

2. **Cape Town, South Africa**: Cape Town is a popular destination for families, with its stunning beaches, vibrant culture, and a rich history.
